# Poglavje 7: Gradnja klepetalnih aplikacij
## Hiter začetek z OpenAI API

Ta zvezek je prilagojen iz [Azure OpenAI vzorčne zbirke](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), ki vključuje zvezke, ki dostopajo do storitev [Azure OpenAI](notebook-azure-openai.ipynb).

Python OpenAI API deluje tudi z Azure OpenAI modeli, z nekaj prilagoditvami. Več o razlikah si preberite tukaj: [Kako preklopiti med OpenAI in Azure OpenAI končnimi točkami s Pythonom](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Pregled  
"Veliki jezikovni modeli so funkcije, ki preslikajo besedilo v besedilo. Glede na vhodni niz besedila velik jezikovni model poskuša napovedati, katero besedilo bo sledilo"(1). Ta zvezek »hitri začetek« bo uporabnikom predstavil osnovne koncepte LLM, osnovne zahteve paketa za začetek z AML, rahel uvod v oblikovanje pozivov in več kratkih primerov različnih primerov uporabe. 


## Kazalo vsebine  

[Pregled](#overview)  
[Kako uporabljati storitev OpenAI](#how-to-use-openai-service)  
[1. Ustvarjanje vaše storitve OpenAI](#1.-creating-your-openai-service)  
[2. Namestitev](#2.-installation)    
[3. Pooblastila](#3.-credentials)  

[Primeri uporabe](#use-cases)    
[1. Povzetek besedila](#1.-summarize-text)  
[2. Razvrščanje besedila](#2.-classify-text)  
[3. Ustvarjanje novih imen izdelkov](#3.-generate-new-product-names)  
[4. Izboljšanje klasifikatorja](#4.fine-tune-a-classifier)  

[Viri](#references)


### Ustvarite svoj prvi poziv  
Ta kratka vaja bo zagotovila osnovni uvod v pošiljanje pozivov modelu OpenAI za preprosto nalogo "povzetek".


**Koraki**:  
1. Namestite knjižnico OpenAI v svoje Python okolje  
2. Naložite standardne pomožne knjižnice in nastavite svoje običajne varnostne poverilnice OpenAI za storitev OpenAI, ki ste jo ustvarili  
3. Izberite model za svojo nalogo  
4. Ustvarite preprost poziv za model  
5. Pošljite svojo zahtevo na API modela!


### 1. Namestite OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Uvozi pomožne knjižnice in ustvari poverilnice


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Iskanje pravega modela  
Modeli, kot sta GPT-4o in GPT-4o mini, razumejo in ustvarjajo naravni jezik ter so na voljo na platformi OpenAI z različnimi nivoji moči in hitrosti, primernimi za različne naloge.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Oblikovanje pozivov  

"Čar velikih jezikovnih modelov je v tem, da se s treniranjem za minimizacijo te napake napovedovanja na ogromnih količinah besedil modeli na koncu naučijo pojmov, uporabnih za te napovedi. Na primer, naučijo se pojmov, kot so"(1):

* kako črkovati
* kako deluje slovnica
* kako parafrazirati
* kako odgovarjati na vprašanja
* kako voditi pogovor
* kako pisati v mnogih jezikih
* kako programirati
* itd.

#### Kako nadzorovati velik jezikovni model  
"Od vseh vhodov velikega jezikovnega modela je daleč najvplivnejši besedilni poziv(1).

Velike jezikovne modele je mogoče pozivati na nekaj načinov:

Navodilo: Povejte modelu, kaj želite
Izpolnitev: Napeljite model, naj dokonča začetek tega, kar želite
Demonstracija: Pokažite modelu, kaj želite, z:
Nekaj primeri v pozivu
Veliko sto ali tisoč primerov v učnem naboru za fino nastavljanje"



#### Obstajajo tri osnovna pravila za ustvarjanje pozivov:

**Pokaži in povej**. Jasno povejte, kaj želite, bodisi preko navodil, primerov ali kombinacije obeh. Če želite, da model razvrsti seznam elementov po abecedi ali da razvrsti odstavek glede na čustveno držo, mu pokažite, da to želite.

**Zagotovite kakovostne podatke**. Če poskušate ustvariti klasifikator ali doseči, da model sledi vzorcu, poskrbite, da bo dovolj primerov. Bodite pozorni na pregled primerov — model je običajno dovolj pameten, da vidi osnovne napake pri črkovanju in vam da odgovor, a lahko tudi predvideva, da je to namenoma in to lahko vpliva na odgovor.

**Preverite nastavitve.** Nastavitve temperature in top_p nadzorujejo, kako determinističen je model pri ustvarjanju odgovora. Če zahtevate odgovor, kjer je le eden pravilen, jih želite nastaviti nižje. Če iščete bolj raznolike odgovore, jih lahko nastavite višje. Največja napaka ljudi pri teh nastavitvah je, da mislijo, da so to kontrola "pametnosti" ali "kreativnosti".


Vir: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Oddaj!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ponovite isti klic, kako se rezultati primerjajo?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Povzetek besedila  
#### Izziv  
Povzemite besedilo tako, da na konec besedilnega odstavka dodate 'tl;dr:'. Opazite, kako model razume, kako izvesti več opravil brez dodatnih navodil. Lahko eksperimentirate z bolj opisnimi pozivi kot tl;dr, da spremenite vedenje modela in prilagodite povzemanje, ki ga prejmete(3).  

Zadnje delo je pokazalo pomembne izboljšave pri številnih nalogah in merilih za obdelavo naravnega jezika z vnaprejšnjim učenjem na velikem korpusu besedila, ki mu sledi fino prilagajanje za določeno nalogo. Čeprav je arhitektura ponavadi neodvisna od naloge, ta metoda še vedno zahteva podatkovne nize za fino prilagajanje, specifične za nalogo, ki vsebujejo tisoče ali desetine tisoč primerov. Nasprotno pa ljudje na splošno lahko opravijo novo jezikovno nalogo že z le nekaj primeri ali preprostimi navodili - nekaj, s čimer se trenutni sistemi za obdelavo naravnega jezika še vedno večinoma težko spopadajo. Tukaj pokažemo, da znatno povečanje velikosti jezikovnih modelov močno izboljša neodvisno, maloštevilsko (few-shot) uspešnost nalog, včasih celo dosega konkurenco s prejšnjimi vrhunskimi pristopi fino prilagajanja.  



Povzeto  


# Vaje za več primerov uporabe  
1. Povzemanje besedila  
2. Razvrščanje besedila  
3. Ustvarjanje novih imen izdelkov


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klasificiraj besedilo  
#### Izziv  
Razvrsti elemente v kategorije, podane ob času sklepanja. V naslednjem primeru zagotovimo tako kategorije kot tudi besedilo za klasifikacijo v pozivu (*playground_reference). 

Povpraševanje stranke: Pozdravljeni, pred kratkim se je na tipkovnici mojega prenosnika zlomila ena tipka in potreboval bom zamenjavo:

Klasificirana kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Ustvari nove izdelke imena
#### Izziv
Ustvarite imena izdelkov iz primerov besed. Tukaj v pozivu vključimo informacije o izdelku, za katerega bomo ustvarili imena. Prav tako zagotovimo podoben primer, da pokažemo vzorec, ki ga želimo prejeti. Nastavili smo tudi visoko vrednost temperature, da povečamo naključnost in bolj inovativne odgovore.

Opis izdelka: Domači aparat za pripravo mlečnih koktajlov
Izvorne besede: hiter, zdrav, kompakten.
Imena izdelkov: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis izdelka: Par čevljev, ki se prilegajo vsaki velikosti stopala.
Izvorne besede: prilagodljiv, prilegajoč, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Reference  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najboljše prakse za fino prilagajanje GPT modelov za razvrščanje besedila](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za več pomoči  
[Ekipa za komercializacijo OpenAI](AzureOpenAITeam@microsoft.com) 


# Prispevalci
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
